# คู่มือการใช้งาน OpenAI API Client

ในการพัฒนา Application ที่เชื่อมต่อกับโมเดล LLMs ในปัจจุบัน เราสามารถเลือกวิธีการเชื่อมต่อได้หลากหลายรูปแบบ ขึ้นอยู่กับความเหมาะสมของงาน ดังนี้:

## 1. **เชื่อมต่อโดยใช้ HTTP client แบบง่ายๆ** 
อย่าง `curl` ใน Command Line หรือ HTTP client `Postman`

```bash
# ตัวอย่างการใช้ curl เรียกใช้งาน Ollama
curl http://localhost:11434/api/generate -d '{
  "model": "gemma3",
  "prompt": "Why is the sky blue?"
}'
```

วิธีนี้เหมาะสำหรับการทดสอบ API เบื้องต้น แต่จะจัดการได้ยากเมื่อโค้ดมีความซับซ้อนขึ้น



## 2. **เชื่อมต่อโดยใช้ Native SDK ที่พัฒนาเฉพาะของแต่ละ AI Platform โดยตรง**

AI Platform แต่ละค่ายมักจะมี Library ของตัวเอง (เช่น ollama-python, google-generativeai) ซึ่งถูกออกแบบมาเพื่อใช้งานฟีเจอร์เฉพาะตัวของค่ายนั้นๆ 

แต่มีข้อเสีย คือ หากต้องการเปลี่ยนค่ายในภายหลัง = อาจจะต้อง **เขียนโค้ดใหม่ทั้งหมด**

```python
# ตัวอย่างการใช้ Ollama Python SDK 

from ollama import chat
from ollama import ChatResponse

response: ChatResponse = chat(model='gemma3', messages=[
  {
    'role': 'user',
    'content': 'Why is the sky blue?',
  },
])
print(response['message']['content'])

```

## 3. **เชื่อมต่อโดยใช้ OpenAI API Client**

#### ทำไมถึงควรใช้ "OpenAI API Client"?

แม้ว่าเราจะไม่ได้ใช้โมเดลของ OpenAI แต่เราก็ยังสามารถใช้ OpenAI API Client ได้ เนื่องจาก OpenAI ได้กลายเป็น _มาตรฐานกลาง_ (De Facto Standard) ในการเชื่อมต่อกับ AI Endpoint ที่ผู้ให้บริการ AI เจ้าอื่นๆ ยอมรับและพัฒนา API ให้รองรับ 

_ค้นหาด้วย keywords: OpenAI Compatibility_

####  ข้อดีของการใช้ OpenAI API Client:
* Write Once, Run Anywhere: เขียนโค้ดเพียงครั้งเดียว แต่สลับไปใช้ AI เจ้าไหนก็ได้ทันที โดยแค่เปลี่ยน `base_url` และ `access token`
* Community Support: มี Library และเครื่องมือเสริมมากมายที่รองรับมาตรฐานนี้

#### ตัวอย่าง AI Platform ที่ OpenAI Compatible
* Gemini OpenAI compatibility: https://ai.google.dev/gemini-api/docs/openai
* Claude: https://platform.claude.com/docs/en/api/openai-sdk
* OpenRouter: https://openrouter.ai/docs/quickstart
* Ollama: https://docs.ollama.com/api/openai-compatibility 

In [ ]:
# /api/tags

# ตัวอย่างการใช้งาน Google Gemini

see: https://ai.google.dev/gemini-api/docs

In [ ]:
GEMINI_API_KEY = ""

In [2]:
from openai import OpenAI

client = OpenAI(
    api_key=GEMINI_API_KEY,
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/"
)

# ทดสอบการใช้งาน OpenAI API Client เบื้องต้น

สามารถเลือกชื่อโมเดลได้ในลิงค์: https://ai.google.dev/gemini-api/docs/models 

In [3]:
response = client.chat.completions.create(
    model="gemini-2.5-flash",
    # max_tokens = 150,
    messages=[
        {"role": "system", "content": "You are a helpful assistant."},
        {
            "role": "user",
            "content": "Explain to me how AI works; keep it short! return plain text without markdown"
        }
    ]
)

In [5]:
print(response.choices[0].message.content)

AI works by using computer programs called algorithms to learn patterns from large amounts of data. These algorithms analyze the data to identify relationships and make predictions or decisions based on what they've learned. It's essentially teaching a computer to recognize things, understand context, or solve problems by showing it many examples, rather than explicitly programming every single rule.




## ตรวจสอบจำนวน Token: 

ทุกๆ Request ระบบจะส่งข้อมูล Usage กลับมาด้วยเสมอ 
* `prompt_tokens`: จำนวน Token ที่เราส่งไป (คำถาม) 
* `completion_tokens`: จำนวน Token ที่ AI สร้างขึ้น (คำตอบ) 
* `total_tokens`: ผลรวมของทั้งสองส่วน 


ข้อสังเกต คือ
```
total_tokens != prompt_tokens + completion_tokens
```

เนื่องจาก มี token บางส่วนถูกใช้ไปในส่วนของ Reasoning/Thinking mode ซึ่งจำนวน token ส่วนนี้จะโดนละไว้เสมอ 

แต่เราสามารถอนุมาณได้ดังนี้

```
total_tokens =  thinking_tokens+ (prompt_tokens + completion_tokens)
```

_NOTE: ในกรณีที่เป็น task ง่ายๆ การไม่ปิด Thinking mode จะทำให้การประมวลช้าและมีค่าใช้จ่ายในส่วนที่ไม่จำเป็นมากขึ้น_

In [7]:
print("Input token", response.usage.prompt_tokens)
print("Output token", response.usage.completion_tokens)
print("Total token", response.usage.total_tokens)

print("Note:", response.usage.completion_tokens+response.usage.prompt_tokens, "!=", response.usage.total_tokens)

Input token 69
Output token 24
Total token 181
Note: 93 != 181


In [8]:
#

# การปรับแต่งพารามิเตอร์

## การปิดโหมด Reasoning/Thinking

**ข้อดี**: ลดเวลาในการรอ (Lower Latency) ตอบกลับได้เร็วขึ้นมาก

**ข้อเสีย**: อาจลดความแม่นยำในงานที่ซับซ้อน หรือ งานที่ต้องใช้การตรรกะ

In [9]:
response = client.chat.completions.create(
    model="gemini-2.5-flash",
    # max_tokens = 150,
    messages=[
        {"role": "system", "content": "You are a helpful assistant."},
        {
            "role": "user",
            "content": "Explain to me how AI works; keep it short! return plain text without markdown"
        }
    ],
    extra_body={
      "extra_body": {
        "google": {
          "thinking_config": {
            "thinking_budget": 0,
            "include_thoughts": False
          }
        }
      }
    }
)



In [10]:
print(response.choices[0].message.content)

AI works by processing vast amounts of data to find patterns and make predictions. It uses algorithms—sets of rules—to learn from this data, effectively training itself. When given new information, it applies what it's learned to perform tasks like recognizing objects, understanding language, or generating text. It's about teaching computers to simulate human-like intelligence through experience.


In [11]:

print("Input token", response.usage.prompt_tokens)
print("Output token", response.usage.completion_tokens)
print("Total token", response.usage.total_tokens)

print("Note:", response.usage.completion_tokens+response.usage.prompt_tokens, "==", response.usage.total_tokens)

Input token 74
Output token 24
Total token 98
Note: 98 == 98


## การตั้งค่า Max Tokens

พารามิเตอร์ `max_tokens` ใช้เพื่อจำกัดจำนวนคำตอบไม่ให้ยาวเกินไป

In [26]:
response = client.chat.completions.create(
    model="gemini-2.5-flash",
    max_tokens = 15,
    messages=[
        {"role": "system", "content": "You are a helpful assistant."},
        {
            "role": "user",
            "content": "Explain to me how AI works; return plain text without markdown markdown markdown markdown"
        }
    ],
    extra_body={
      "extra_body": {
        "google": {
          "thinking_config": {
            "thinking_budget": 0,
            "include_thoughts": False
          }
        }
      }
    }
)

In [27]:
print(response.choices[0].message.content)

Imagine a super-smart computer program. That's essentially what AI is


In [28]:
print("Input token", response.usage.prompt_tokens)
print("Output token", response.usage.completion_tokens)
print("Total token", response.usage.total_tokens)

print("SEE!!", response.usage.completion_tokens+response.usage.prompt_tokens, "==", response.usage.total_tokens)

Input token 23
Output token 15
Total token 38
SEE!! 38 == 38


#### การตรวจสอบ `finish_reason`

AI สามารถหยุดทำงานได้ด้วยหลายเหตุผล หากมีการตั้งค่า `max_token` แนะนำให้ตรวจสอบ `finish_reason` ด้วยทุกครั้ง

In [21]:
response.choices[0].finish_reason

'length'

ตัวอย่างค่าที่เป็นไปได้

* `stop` if กรณีที่โมเดลหยุดที่จุดสิ้นสุดปกติ หรือ หยุดเมื่อเจอเงื่อนไขการหยุดที่เรากำหนดไว้
* `length` if กรณีที่จำนวน Token ถึงขีดจำกัดสูงสุด ตามที่ระบุไว้ใน request
* `content_filter` if กรณีที่เนื้อหาถูกตัดออกเนื่องจากตัวกรองเนื้อหา (Content filters) ของระบบ
* `tool_calls` if กรณีที่โมเดลรอผลลัพท์จากการเรียกใช้งานเครื่องมือ
* `function_call` (deprecated)

หากใช้ google.genai (Gemini SDK ของ google) 

จะมี `finish_reason` ที่ละเอียดมากขึ้น ตามรูปด้านล่าง

In [32]:
from IPython.display import Image
from IPython.core.display import HTML 
Image(url= "./Images/FinishReasons.png")

## การทำให้ผลลัพธ์คงที่ (Make it Deterministic)

คำตอบที่ได้จาก AI โดยทั่วไปจะตอบไม่เหมือนเดิมในแต่ละครั้ง เพราะการคำนวนของ LLMs จะมีการสุ่มเข้ามาเกี่ยวข้องด้วย 

ทั้งนี้เราสามารถควบคุมให้ AI มีคำตอบคงที่ได้ด้วย ตัวแปรที่เรียกว่า `temperature`

see: https://docs.cloud.google.com/vertex-ai/generative-ai/docs/learn/prompts/adjust-parameter-values#temperature

#### ตัวอย่างกรณีที่ไม่ได้ตั้งค่า temperature

In [22]:
def run():
    return client.chat.completions.create(
        model="gemini-2.5-flash",
        max_tokens = 500,
        messages=[
            {"role": "system", "content": "You are a helpful assistant."},
            {
                "role": "user",
                "content": "Explain to me how AI works in one sentence. Return plain text without markdown."
            }
        ],
        extra_body={
          "extra_body": {
            "google": {
              "thinking_config": {
                "thinking_budget": 0,
                "include_thoughts": False
              }
            }
          }
        }
    )

print("Output #1:")
response = run()
print(response.choices[0].message.content)

print()
print()
print("Output #2:")
response = run()
print(response.choices[0].message.content)

Output #1:
AI works by using algorithms to identify patterns in data, learn from them, and make decisions or predictions.


Output #2:
AI works by using algorithms to identify patterns and make decisions based on data.


#### ตัวอย่างกรณีที่ตั้งค่า temperature = 0

Default: `tempurature = 1.0`

NOTE: 
* lower temperatures are good for prompts that require a less open-ended or creative response
* higher temperatures can lead to more diverse or creative results.

ในกรณีที่เซต temperature ต่ำๆ อาจจะเกิดปัญหาแปลกๆ เช่น โมเดลให้คำตอบวนๆซ้ำไปซ้ำมา หรือ ทำงานใน thinking mode ได้แย่ลงมากๆ

In [35]:
def run():
    return client.chat.completions.create(
        model="gemini-2.5-flash",
        temperature = 0,
        max_tokens = 500,
        messages=[
            {"role": "system", "content": "You are a helpful assistant."},
            {
                "role": "user",
                "content": "Explain to me how AI works in one sentence. Return plain text without markdown."
            }
        ],
        extra_body={
          "extra_body": {
            "google": {
              "thinking_config": {
                "thinking_budget": 0,
                "include_thoughts": False
              }
            }
          }
        }
    )

print("Output #1:")
response = run()
print(response.choices[0].message.content)

print()
print()
print("Output #2:")
response = run()
print(response.choices[0].message.content)

Output #1:
AI works by using algorithms to process data, learn patterns, and make decisions or predictions.


Output #2:
AI works by using algorithms to process data, learn patterns, and make decisions or predictions.


# การใช้งานผ่าน Google GenAI SDK (Native Client)

นอกจากการใช้ OpenAI Client แล้ว Google มี AI SDK เฉพาะของตัวเอง ชื่อว่า `google-genai`  https://github.com/googleapis/python-genai

#### ข้อดี
เป็น Client เฉพาะทางที่รองรับฟีเจอร์ใหม่ๆ ของ Gemini ได้เร็วกว่า และมีการจัดการโครงสร้างข้อมูลที่ปรับมาเพื่อโมเดลตระกูล Gemini โดยเฉพาะ หรือการปรับแต่งระบบความปลอดภัย (Safety Settings) ที่ละเอียดกว่าปกติ

In [37]:
# !uv pip install google-genai

In [41]:
from google import genai
from google.genai import types

client = genai.Client(api_key=GEMINI_API_KEY)

response = client.models.generate_content(
    model="gemini-2.5-flash",
    contents="อธิบายหลักการทำงานของ AI สั้นๆ เป็นภาษาไทย โดยไม่ต้องใช้ Markdown",
    config=types.GenerateContentConfig(
        # max_output_tokens=100 
    )
)

In [42]:
print(response.text)

AI คือโปรแกรมคอมพิวเตอร์ที่ถูกสร้างให้สามารถ "เรียนรู้" ได้เหมือนมนุษย์ในระดับหนึ่ง

หลักการทำงานง่ายๆ คือ:
1.  **ป้อนข้อมูล:** AI จะได้รับข้อมูลจำนวนมากและหลากหลาย (เช่น รูปภาพ, ข้อความ, ตัวเลข) เพื่อใช้ในการฝึกฝน
2.  **เรียนรู้รูปแบบ:** มันจะวิเคราะห์ข้อมูลเหล่านั้นเพื่อหารูปแบบ ความสัมพันธ์ และกฎเกณฑ์ต่างๆ ด้วยอัลกอริทึม
3.  **นำไปใช้:** เมื่อได้รับข้อมูลใหม่ที่ยังไม่เคยเห็น AI ก็จะใช้รูปแบบที่เรียนรู้มานี้มา "คาดเดา" "ตัดสินใจ" หรือ "ทำงานบางอย่าง" ได้ (เช่น จดจำใบหน้า, แปลภาษา, แนะนำสินค้า)

สรุปคือ AI เรียนรู้จากข้อมูล หาสิ่งที่เชื่อมโยงกัน แล้วนำไปใช้ตอบโจทย์ใหม่ๆ ยิ่งข้อมูลเยอะและดีเท่าไหร่ AI ก็จะยิ่งฉลาดและแม่นยำขึ้นเท่านั้น


In [50]:
print("Input token", response.usage_metadata.prompt_token_count)
print("Thought token", response.usage_metadata.thoughts_token_count)
print("Output token", response.usage_metadata.candidates_token_count)

print("Total token", response.usage_metadata.total_token_count)
print(f"Finish Reason: {response.candidates[0].finish_reason}")

Input token 18
Thought token 1261
Output token 210
Total token 1489
Finish Reason: FinishReason.STOP


In [51]:
print(response.usage_metadata.prompt_tokens_details)

[ModalityTokenCount(
  modality=<MediaModality.TEXT: 'TEXT'>,
  token_count=18
)]
